# A general grammar checker — Qwen3-0.6B

Two decisions here come from measurements on the previous banking build, not
from taste:

**The model emits the corrected sentence, not an edit list.** The banking model
was trained to output `[{"o": ..., "c": ..., "n": ...}]`, which forces it to
reproduce exact source spans; one wrong boundary destroys the whole correction.
It scored **51.79 GLEU** on JFLEG against **55.60** for `coedit-large`, an
off-the-shelf model that simply writes the corrected sentence. Producing text is
a strictly easier task. Production still gets JSON edits — they are *derived*
from the corrected text with `derive_edits`, the same function that built the
gold labels, so they are computed rather than guessed.

**All banking data is gone.** In the previous run 94,000 of 215,000 correction
examples were synthetic banking corruptions from nine hand-written functions.
For a general checker that is not training signal, it is noise.

The whole budget now goes to grammatical error correction data, most of it
human-annotated.

## 0. Environment

In [ ]:
import os, shutil, subprocess, re as _re
from pathlib import Path

# Pinned on purpose. LLaMA-Factory `main` and the latest transformers drift
# apart: on 2026-11-20 they broke this notebook twice (transformers 5.x dropped
# `warmup_ratio`; transformers>=5.16 passes `every_n_layers` to a
# gradient-checkpointing hook LLaMA-Factory overrides without that argument).
LF_COMMIT = "b4e4d3fcb2a5a13ce6b9431d395e3939f84b8ec1"

if not Path("LLaMA-Factory").exists():
    subprocess.run(["git", "init", "-q", "LLaMA-Factory"], check=True)
    subprocess.run(["git", "-C", "LLaMA-Factory", "fetch", "-q", "--depth", "1",
                    "https://github.com/hiyouga/LLaMA-Factory.git", LF_COMMIT], check=True)
    subprocess.run(["git", "-C", "LLaMA-Factory", "checkout", "-q", "FETCH_HEAD"], check=True)

%cd LLaMA-Factory
%pip install -q -e ".[torch,metrics]"
%cd ..

%pip install -q datasets "transformers==5.15.1" accelerate sentencepiece pyyaml psutil pandas pyarrow

In [ ]:
import json, re, random, difflib, time, gc, math
from collections import Counter
import pandas as pd, psutil, torch

SEED = 13
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Re-running will NOT reproduce these weights bit for bit. Seeding fixes data
# order, initialisation and dropout; it does not fix fp16 accumulation order or
# cuDNN kernel selection, which vary run to run on GPU. Expect the same pipeline
# to land within roughly a point of GLEU, not on the exact number. If a specific
# checkpoint matters, save the weights -- do not plan to recreate them.

ROOT     = Path("gec")
DATA_DIR = ROOT / "data"
CFG_DIR  = ROOT / "configs"
SAVE_DIR = ROOT / "saves"
for d in (DATA_DIR, CFG_DIR, SAVE_DIR):
    d.mkdir(parents=True, exist_ok=True)

LF_DIR = Path("LLaMA-Factory")
BASE_MODEL = "Qwen/Qwen3-0.6B"

_T0 = time.time()
def stage(m): print(f"[stage +{time.time()-_T0:7.1f}s] {m}", flush=True)

# ---- run mode -----------------------------------------------------------
SMOKE = False

if SMOKE:
    N_C4, N_WI, N_COEDIT = 1_500, 1_500, 1_000
    MAX_STEPS = 60
else:
    # Lang-8 was the plan for bulk human-annotated data, but every English
    # mirror on the Hub is now gated (401) -- the datasets-server preview that
    # made it look available was serving a cache. So: take ALL of W&I+LOCNESS
    # and CoEdIT, and fill the rest with C4-200M. C4-200M is generated rather
    # than human-annotated, but by a learned corruption model trained on real
    # errors -- it is the standard GEC pretraining corpus, not rule-based noise
    # like the banking corruptions this build removed.
    # ~220k examples at the measured 8.2 samples/s -> ~7.4 h of training.
    N_C4, N_WI, N_COEDIT = 140_000, 38_000, 18_000
    MAX_STEPS = None

NOOP_RATIO = 0.12
MAX_SRC_CHARS = 300

print("SMOKE run" if SMOKE else "FULL run")
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    arches = {int(a.split("_")[1]) for a in torch.cuda.get_arch_list() if a.startswith("sm_")}
    for i in range(torch.cuda.device_count()):
        major, minor = torch.cuda.get_device_capability(i)
        cap = major * 10 + minor
        if cap not in arches:
            raise RuntimeError(
                f"GPU {i} ({torch.cuda.get_device_name(i)}) is sm_{cap}, but torch "
                f"{torch.__version__} only has kernels for {sorted(arches)}. "
                "Set the accelerator to 'GPU T4 x2' and re-run.")
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

print("CPU cores:", os.cpu_count(), "| RAM:",
      round(psutil.virtual_memory().total / 1024**3, 1), "GiB")

## 1. Edit derivation — used for the gold labels and for production JSON

In [ ]:
WORD_RE = re.compile(r"\S+")
def toks(s): return WORD_RE.findall(s)
def norm(s): return re.sub(r"\s+", " ", s).strip()

def derive_edits(src, tgt):
    """Word-level diff -> compact edit list.

    Occurrence index `n` is counted in *substring* space, because apply_edits
    resolves spans with str.find(). Getting that wrong silently breaks any
    sentence containing a repeated word ("the" also occurs inside "there").
    """
    spans = [(m.group(0), m.start(), m.end()) for m in WORD_RE.finditer(src)]
    a = [s[0] for s in spans]
    b = toks(tgt)
    edits = []
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(a=a, b=b, autojunk=False).get_opcodes():
        if tag == "equal":
            continue
        if tag == "insert":
            if i1 > 0:      k, pre = i1 - 1, True
            elif i1 < len(a): k, pre = i1, False
            else:            return None
            cs, ce = spans[k][1], spans[k][2]
            o = src[cs:ce]
            ins = " ".join(b[j1:j2])
            c = (o + " " + ins) if pre else (ins + " " + o)
        else:
            cs, ce = spans[i1][1], spans[i2 - 1][2]
            o = src[cs:ce]
            c = " ".join(b[j1:j2])
        edits.append({"o": o, "c": c, "n": src[:cs].count(o) + 1})
    return edits or None


def apply_edits(src, edits):
    """Strict inverse of derive_edits. Used to validate gold, where an edit list
    that cannot round-trip should be rejected outright."""
    out = src
    for e in edits:
        o, c, n = e["o"], e["c"], e.get("n", 1)
        idx, start = -1, 0
        for _ in range(n):
            idx = out.find(o, start)
            if idx == -1:
                return None
            start = idx + 1
        out = out[:idx] + c + out[idx + len(o):]
    return norm(out)


def roundtrips(src, tgt):
    e = derive_edits(src, tgt)
    if e is None:
        return None
    got = apply_edits(src, e)
    return e if got is not None and got == norm(tgt) else None


_s = "The account holder were notified of the changes to there statement."
_t = "The account holder was notified of the changes to their statement."
print(json.dumps(roundtrips(_s, _t), ensure_ascii=False))

## 2. Data — grammatical error correction only

Four corpora, three of them human-annotated. Every pair is kept only if
`derive_edits` can round-trip it, so a corrupted or unalignable pair never
reaches training.

In [ ]:
stage("loading GEC corpora")

from datasets import load_dataset

# (tag, repo, source column, target column, wanted, human-annotated?)
SOURCES = [
    ("wi_locness", "512duncanl/wi_locness_detokenized", "input",  "output", N_WI,     True),
    ("c4_200m",    "512duncanl/c4_200m_cleaned_365k",   "input",  "output", N_C4,     False),
]

pairs = {}

for tag, repo, ci, co, want, human in SOURCES:
    got = []
    try:
        ds = load_dataset(repo, split="train", streaming=True)
        scanned = 0
        for row in ds:
            scanned += 1
            if scanned > want * 30:
                print(f"  {tag}: exhausted scan budget at {scanned:,} rows")
                break
            s = (row.get(ci) or "").strip()
            t = (row.get(co) or "").strip()
            if not s or not t or s == t or len(s) > MAX_SRC_CHARS:
                continue
            if roundtrips(s, t) is None:
                continue
            got.append((s, t))
            if len(got) >= want:
                break
    except Exception as ex:
        print(f"  {tag} unavailable: {ex}")
    pairs[tag] = got
    print(f"{tag:<12} {len(got):>7,} pairs   {'human-annotated' if human else 'auto-generated'}")

# CoEdIT ships many tasks; only the gec ones are corrections rather than rewrites.
try:
    ce = load_dataset("grammarly/coedit", split="train")
    got = []
    for row in ce:
        if row.get("task") != "gec":
            continue
        s = re.sub(r"^[^:]*:\s*", "", (row.get("src") or "")).strip()
        t = (row.get("tgt") or "").strip()
        if not s or not t or s == t or len(s) > MAX_SRC_CHARS:
            continue
        if roundtrips(s, t) is None:
            continue
        got.append((s, t))
        if len(got) >= N_COEDIT:
            break
    pairs["coedit"] = got
except Exception as ex:
    print("  coedit unavailable:", ex)
    pairs["coedit"] = []
print(f"{'coedit':<12} {len(pairs['coedit']):>7,} pairs   human-annotated")

total = sum(len(v) for v in pairs.values())
human = sum(len(pairs[t]) for t in ("wi_locness", "coedit"))
print(f"\ntotal correction pairs: {total:,}")
print(f"human-annotated share : {human/max(total,1):.1%}")
if not total:
    raise RuntimeError("no GEC data loaded")

## 3. Training records — corrected text as the target

The previous build made the model emit an edit list. This one makes it emit the
corrected sentence; the edit list is derived afterwards. Clean sentences are
mixed in at `NOOP_RATIO` so the model learns that "leave it alone" is frequently
the right answer — without them a checker rewrites text that was already fine.

In [ ]:
stage("building training records")

INSTRUCTION = (
    "Correct the grammar, spelling, and punctuation of the sentence. "
    "Do not change its meaning or style. "
    "Reply with the corrected sentence only. "
    "If it is already correct, repeat it unchanged."
)

def to_alpaca(src, tgt, tag):
    return {"instruction": INSTRUCTION, "input": src, "output": tgt, "_tag": tag}

records = []
for tag, plist in pairs.items():
    for s, t in plist:
        records.append(to_alpaca(s, norm(t), tag))

# no-op examples: already-correct sentences drawn from the targets
clean = [norm(t) for plist in pairs.values() for _, t in plist]
random.shuffle(clean)
n_noop = int(len(records) * NOOP_RATIO)
for t in clean[:n_noop]:
    records.append(to_alpaca(t, t, "noop"))

random.shuffle(records)
print(Counter(r["_tag"] for r in records))
print(f"total records: {len(records):,}")

## 4. Dedupe, split, Parquet

Drop exact duplicates, then split train/dev on a **hash of the content** rather
than a random shuffle: a given example always lands on the same side, so adding
more data later can never move a training example into dev.

LLaMA-Factory reads a single `.parquet` file directly, so there is no directory
of part-files to clean up.


In [ ]:
stage("dedupe, split, write")

import hashlib

df = pd.DataFrame(records)
before = len(df)
df = df.drop_duplicates(subset=["input", "output"]).reset_index(drop=True)
print(f"deduped: {before:,} -> {len(df):,}")
print(df["_tag"].value_counts().to_string())

# Content hash, not random: an example always lands on the same side, so adding
# data later cannot move a training example into dev.
def bucket(inp, out):
    h = hashlib.md5(f"{inp}\x00{out}".encode("utf-8")).hexdigest()
    return int(h[:8], 16) % 100

buckets = [bucket(i, o) for i, o in zip(df["input"], df["output"])]
df = df.assign(_bucket=buckets)

# Sort before writing: dedupe order depends on input order, and the trainer's
# shuffle is a permutation of row positions -- an unstable order means different
# batches on every run, for no reason.
df = df.sort_values(["input", "output"], kind="mergesort").reset_index(drop=True)

cols = ["instruction", "input", "output"]
train = df.loc[df["_bucket"] < 98, cols]
dev_df = df.loc[df["_bucket"] >= 98, cols]

TRAIN_PARQUET = DATA_DIR / "gec_train.parquet"
DEV_PARQUET   = DATA_DIR / "gec_dev.parquet"
train.to_parquet(TRAIN_PARQUET, index=False)
dev_df.to_parquet(DEV_PARQUET, index=False)

print(f"train: {len(train):,}  |  dev: {len(dev_df):,}")

dev = dev_df.to_dict("records")
print("dev collected:", len(dev))

In [ ]:
info_path = LF_DIR / "data" / "dataset_info.json"
info = json.loads(info_path.read_text())
for name, path in [("gec_train", TRAIN_PARQUET), ("gec_dev", DEV_PARQUET)]:
    info[name] = {
        "file_name": str(path.resolve()),
        "columns": {"prompt": "instruction", "query": "input", "response": "output"},
    }
info_path.write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding="utf-8")
print("registered gec_train / gec_dev")

## 5. Fine-tune

In [ ]:
import yaml

cfg = {
    "model_name_or_path": BASE_MODEL,
    "trust_remote_code": True,
    "seed": 13,          # HF Trainer defaults to 42; pin it with everything else
    "stage": "sft",
    "do_train": True,
    "finetuning_type": "full",

    "dataset": "gec_train",
    "eval_dataset": "gec_dev",
    # llamafactory-cli inherits this notebook's CWD, but dataset_info.json lives
    # inside the cloned repo; without this it looks for ./data/dataset_info.json.
    "dataset_dir": str((LF_DIR / "data").resolve()),
    "template": "qwen3",
    "enable_thinking": False,
    "cutoff_len": 512,
    "overwrite_cache": True,
    "preprocessing_num_workers": min(8, os.cpu_count() or 4),
    "dataloader_num_workers": min(4, os.cpu_count() or 2),

    "output_dir": str((SAVE_DIR / "qwen3-0.6b-gec").resolve()),
    "logging_steps": 50,
    "save_steps": 2000,
    "save_total_limit": 2,
    "plot_loss": True,
    "overwrite_output_dir": True,
    "report_to": "none",

    # 32-bit Adam needs ~13.5 GiB for a full fine-tune of this model and OOMs a
    # 14.56 GiB T4 at any batch size. Adafactor's factored second moment frees
    # ~6 GiB and is built into transformers.
    "optim": "adafactor",
    "per_device_train_batch_size": 8,
    "gradient_accumulation_steps": 2,
    "learning_rate": 2.0e-5,
    "num_train_epochs": 1.0,
    "lr_scheduler_type": "cosine",
    # warmup_steps is set below from the real step count
    "fp16": True,
    "bf16": False,
    "gradient_checkpointing": True,
    "flash_attn": "disabled",
    "ddp_timeout": 1800,

    "per_device_eval_batch_size": 8,
    "eval_strategy": "steps",
    "eval_steps": 1000,
}

if SMOKE:
    cfg.update({"max_steps": MAX_STEPS, "logging_steps": 10,
                "eval_steps": 30, "save_steps": MAX_STEPS})

# transformers 5.x dropped `warmup_ratio` (LLaMA-Factory now installs up to
# 5.17), while older releases read a float `warmup_steps` as a step count. An
# integer step count means the same thing in every version, so derive 3% of the
# real schedule from the training parquet.
import math
import pyarrow.parquet as pq

WARMUP_RATIO = 0.03

def parquet_rows(path):
    path = Path(path)
    if not path.is_absolute():
        path = LF_DIR / "data" / path
    files = sorted(path.rglob("*.parquet")) if path.is_dir() else [path]
    return sum(pq.ParquetFile(f).metadata.num_rows for f in files)

_info = json.loads((LF_DIR / "data" / "dataset_info.json").read_text())
n_train = parquet_rows(_info[cfg["dataset"]]["file_name"])
# Single GPU (TRAIN_GPUS = 1 in the launch cell), so no world-size factor.
steps_per_epoch = math.ceil(n_train / (cfg["per_device_train_batch_size"]
                                       * cfg["gradient_accumulation_steps"]))
total_steps = cfg.get("max_steps") or math.ceil(steps_per_epoch * cfg["num_train_epochs"])
cfg["warmup_steps"] = max(1, round(WARMUP_RATIO * total_steps))
print(f"{n_train:,} train rows -> {total_steps:,} optimizer steps, warmup_steps={cfg['warmup_steps']}")

cfg_path = CFG_DIR / "qwen3_gec.yaml"
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print(cfg_path.read_text())

In [ ]:
stage("fine-tuning")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# MEASURED on this image (torch 2.10 + transformers 5.0 + LLaMA-Factory dev):
# 2-GPU DDP deadlocks in the first backward pass and never emits a step, with or
# without ddp_find_unused_parameters. Single-GPU trains the same steps in 84 s.
TRAIN_GPUS = 1
VISIBLE = ",".join(str(i) for i in range(TRAIN_GPUS))
print("training on CUDA devices:", VISIBLE)

env = dict(os.environ)
env["CUDA_VISIBLE_DEVICES"] = VISIBLE
env["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
env.pop("FORCE_TORCHRUN", None)

TRAIN_TIMEOUT_S = 25 * 60 if SMOKE else 9 * 60 * 60

# `!cmd` swallows the exit status: a failed run would otherwise resurface as a
# baffling "no model.safetensors" error three cells later.
try:
    proc = subprocess.run(["llamafactory-cli", "train", str(cfg_path)],
                          env=env, timeout=TRAIN_TIMEOUT_S)
except subprocess.TimeoutExpired:
    raise RuntimeError(f"training exceeded {TRAIN_TIMEOUT_S/60:.0f} min and was killed")

if proc.returncode != 0:
    raise RuntimeError(f"llamafactory-cli train failed with exit code {proc.returncode}")

FINAL_MODEL = cfg["output_dir"]
print("training finished:", FINAL_MODEL)

## 6. Evaluation — JFLEG, the number that is comparable

GLEU is implemented from the paper (not `nltk`'s Google-BLEU, which ignores the
source sentence — the whole point of GLEU is to penalise n-grams a system left
unchanged that the references changed). It is checked against the published
anchors before it scores anything, and the run aborts if it disagrees.

In [ ]:
def ngrams(t, n):
    return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1))

def gleu_corpus(sources, hyps, refs_list, N=4):
    n_refs = min(len(r) for r in refs_list)
    per_ref = []
    for j in range(n_refs):
        num=[0]*(N+1); den=[0]*(N+1); hl=rl=0
        for src, hyp, refs in zip(sources, hyps, refs_list):
            s,h,r = src.split(), hyp.split(), refs[j].split()
            hl += len(h); rl += len(r)
            for n in range(1, N+1):
                hg, sg, rg = ngrams(h,n), ngrams(s,n), ngrams(r,n)
                hit = sum((hg & rg).values())
                kept_wrong = sum((hg & sg).values()) - sum((hg & sg & rg).values())
                num[n] += max(hit - kept_wrong, 0)
                den[n] += sum(hg.values())
        logs = []
        for n in range(1, N+1):
            p = (num[n]/den[n]) if den[n] else 0.0
            logs.append(math.log(p) if p > 0 else -9e9)
        bp = 1.0 if hl > rl else math.exp(1 - rl/max(hl,1))
        per_ref.append(bp * math.exp(sum(logs)/N))
    return 100 * sum(per_ref) / len(per_ref)


def retokenize(s):
    """JFLEG is Moses-tokenised and GLEU scores those tokens. A system that
    emits normal detokenised prose loses n-gram matches on formatting alone."""
    s = re.sub(r"\s*([,;:!?])", r" \1", s)
    s = re.sub(r"(?<!\d)\s*\.(?!\d)", r" .", s)
    s = re.sub(r"\s*'\s*(s|t|re|ve|ll|d|m)\b", r" '\1", s)
    return norm(s)


stage("loading JFLEG + verifying the metric")
JF = {s: load_dataset("jhu-clsp/jfleg", split=s) for s in ("validation", "test")}
JDATA = {}
for split, ds in JF.items():
    src  = [s.strip() for s in ds["sentence"]]
    refs = [[r.strip() for r in c] for c in ds["corrections"]]
    keep = [i for i, s in enumerate(src) if s]
    JDATA[split] = ([src[i] for i in keep], [refs[i] for i in keep])

for split, (src, refs) in JDATA.items():
    base = gleu_corpus(src, src, refs)
    pub  = {"test": 40.54, "validation": 38.21}[split]
    hum  = gleu_corpus(src, [r[0] for r in refs], [r[1:] for r in refs])
    print(f"{split:<11} n={len(src)}  source={base:6.2f} (published {pub})  human={hum:6.2f}")
    assert abs(base - pub) < 1.5, "GLEU implementation disagrees with the published baseline"
print("metric verified")

In [ ]:
stage("loading models")

from transformers import AutoTokenizer, AutoModelForCausalLM, T5ForConditionalGeneration

ft_tok = AutoTokenizer.from_pretrained(FINAL_MODEL)
ft_model = AutoModelForCausalLM.from_pretrained(
    FINAL_MODEL, dtype=torch.float16,
    device_map={"": 0} if torch.cuda.is_available() else None)
ft_model.eval()
if ft_tok.pad_token_id is None:
    ft_tok.pad_token = ft_tok.eos_token
ft_tok.padding_side = "left"

@torch.no_grad()
def ft_correct(texts, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        chunk = texts[i:i+bs]
        # LLaMA-Factory joins instruction and input with "\n" (converter.py:100).
        prompts = [ft_tok.apply_chat_template(
            [{"role": "user", "content": INSTRUCTION + "\n" + t}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False)
            for t in chunk]
        enc = ft_tok(prompts, return_tensors="pt", padding=True,
                     truncation=True, max_length=512).to(ft_model.device)
        gen = ft_model.generate(**enc, max_new_tokens=160, do_sample=False,
                                pad_token_id=ft_tok.pad_token_id)
        out += [norm(s) for s in ft_tok.batch_decode(
            gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
    return out

bl_tok = AutoTokenizer.from_pretrained("grammarly/coedit-large")
bl_model = T5ForConditionalGeneration.from_pretrained("grammarly/coedit-large")
if torch.cuda.is_available():
    bl_model = bl_model.cuda().half()
bl_model.eval()

@torch.no_grad()
def bl_correct(texts, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        batch = ["Fix grammatical errors in this sentence: " + t for t in texts[i:i+bs]]
        enc = bl_tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=256)
        enc = {k: v.to(bl_model.device) for k, v in enc.items()}
        gen = bl_model.generate(**enc, max_length=256, num_beams=1)
        out += [norm(s) for s in bl_tok.batch_decode(gen, skip_special_tokens=True)]
    return out

In [ ]:
stage("scoring JFLEG")

rows = []
JPRED = {}
for split, (src, refs) in JDATA.items():
    ft = ft_correct(src)
    bl = bl_correct(src)
    JPRED[split] = (src, ft, bl, refs)
    rows.append({
        "split": split, "n": len(src),
        "source": gleu_corpus(src, src, refs),
        "coedit tok": gleu_corpus(src, [retokenize(x) for x in bl], refs),
        "qwen3-ft tok": gleu_corpus(src, [retokenize(x) for x in ft], refs),
        "human": gleu_corpus(src, [r[0] for r in refs], [r[1:] for r in refs]),
        "ft edited %": 100*sum(1 for a,b in zip(src,ft) if norm(a)!=b)/len(src),
    })

print("\n" + "="*84)
print("JFLEG — GLEU (higher is better), scored in the corpus tokenisation")
print("="*84)
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.2f}"))
print("\nPublished anchors: source ~40.5 (test), human ~62.")
print("Previous edit-JSON model scored 51.79 on test.")

## 7. Held-out dev — precision, recall, and how often it touches correct text

`noop_rate` is the number a checker lives or dies by: how often it rewrites a
sentence that was already fine. Both systems are measured on the same event —
did the text actually change — so the comparison is like for like.

In [ ]:
stage("scoring held-out dev")

def score_edits(gold, pred):
    g = {(e["o"], e["c"]) for e in gold}
    p = {(e["o"], e["c"]) for e in pred}
    return len(g & p), len(p - g), len(g - p)

def prf(tp, fp, fn, beta=0.5):
    P = tp/(tp+fp) if tp+fp else 0.0
    R = tp/(tp+fn) if tp+fn else 0.0
    b2 = beta**2
    F = ((1+b2)*P*R/(b2*P+R)) if (P and R) else 0.0
    return P, R, F

N_EVAL = 60 if SMOKE else 400
dev_gec  = [(r["input"], norm(r["output"])) for r in dev if norm(r["input"]) != norm(r["output"])][:N_EVAL]
dev_noop = [r["input"] for r in dev if norm(r["input"]) == norm(r["output"])][:N_EVAL]
print(f"dev: {len(dev_gec)} with errors | {len(dev_noop)} already correct")

def evaluate(correct_fn):
    srcs = [s for s, _ in dev_gec]
    outs = correct_fn(srcs)
    acc, exact = [0,0,0], 0
    for (s, tgt), o in zip(dev_gec, outs):
        gold = derive_edits(s, tgt) or []
        pred = derive_edits(s, o) or []
        a,b,c = score_edits(gold, pred)
        acc[0]+=a; acc[1]+=b; acc[2]+=c
        exact += (o == tgt)
    P,R,F = prf(*acc)
    clean_out = correct_fn(dev_noop)
    touched = sum(1 for a,b in zip(dev_noop, clean_out) if norm(a) != b)
    return {"P": P, "R": R, "F0.5": F,
            "exact_match": exact/len(dev_gec),
            "noop_rate": touched/len(dev_noop)}

FT_DEV = evaluate(ft_correct)
BL_DEV = evaluate(bl_correct)

print("\n" + "="*66)
print("HELD-OUT DEV")
print("="*66)
print(f"{'metric':<16}{'coedit-large':>16}{'qwen3-0.6b-ft':>18}")
print("-"*66)
for k in ("P","R","F0.5","exact_match","noop_rate"):
    print(f"{k:<16}{BL_DEV[k]:>16.3f}{FT_DEV[k]:>18.3f}")

## 8. Production output

The model writes plain corrected text; the JSON edit list a UI needs is derived
from it deterministically, so a span can never be hallucinated — every `o` is by
construction a real substring of the input.

In [ ]:
def check(sentence):
    """Corrected text plus the edit list, ready for a UI."""
    fixed = ft_correct([sentence])[0]
    if fixed == norm(sentence):
        return {"corrected": fixed, "edits": []}
    return {"corrected": fixed, "edits": derive_edits(sentence, fixed) or []}


for s in [
    "I has recieved you're letter yesterday and i will responded soon.",
    "The report were submitted on time and contain no error.",
    "This sentence is already perfectly fine.",
]:
    print(json.dumps(check(s), ensure_ascii=False, indent=2))
    print()

src, ft, bl, refs = JPRED["test"]
pd.DataFrame({"src": src, "qwen3_ft": ft, "coedit_large": bl,
              "ref1": [r[0] for r in refs]}).to_csv("jfleg_predictions.csv", index=False)
print("saved jfleg_predictions.csv")